# Objetivo 2 · Pronóstico: genético frente a Elastic Net y edad
**Reto CIMAT / BraTS 2018** — análisis de los resultados de `pronostico/evaluar.py` (diseño en D27).

Se corre **localmente**, en el entorno `radiomica`; no necesita GPU. Primero hay que generar los resultados:
```
python pronostico/evaluar.py                     # principal
python pronostico/evaluar.py --variante manual   # y los secundarios
```
Cada punto de los gráficos es un fold externo: 5 folds × 10 repeticiones = 50. Los tres brazos usan los mismos folds, así que las diferencias son pareadas.

In [ ]:
import os
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

if Path.cwd().name == 'pronostico':
    os.chdir('..')
BASE = Path('resultados/pronostico')
ETIQUETA = 'pred'          # 'pred' (principal), 'manual', 'pred_sinfiltro', 'pred_reseccion'
R = BASE / ETIQUETA
print('analisis disponibles:', sorted(p.name for p in BASE.iterdir() if (p / 'por_fold.csv').exists()))

pf = pd.read_csv(R / 'por_fold.csv')
sel = pd.read_csv(R / 'seleccion.csv')
curvas = pd.read_csv(R / 'curvas_genetico.csv')

BRAZOS = ['edad', 'coxnet', 'genetico']
NOMBRE = {'edad': 'Edad sola', 'coxnet': 'Cox Elastic Net', 'genetico': 'Genético + Cox'}
COLOR = {'edad': '#2a78d6', 'coxnet': '#eb6834', 'genetico': '#1baf7a'}   # paleta validada (dataviz)
TINTA, TINTA2, REJILLA = '#0b0b0b', '#52514e', '#e5e5e2'
plt.rcParams.update({'axes.edgecolor': TINTA2, 'axes.labelcolor': TINTA2, 'xtick.color': TINTA2,
                     'ytick.color': TINTA2, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': REJILLA, 'grid.linewidth': 0.8, 'font.size': 11})
print((R / 'resumen.txt').read_text())

## 1. c-index por fold externo
La línea gris marca el azar (0.5). La barra horizontal de cada brazo es la mediana.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
rng = np.random.default_rng(0)
for i, b in enumerate(BRAZOS):
    y = pf.loc[pf.brazo == b, 'cindex'].to_numpy()
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(y)), y, s=36, color=COLOR[b], alpha=0.8,
               edgecolor='white', linewidth=1)
    ax.hlines(np.median(y), i - 0.25, i + 0.25, color=TINTA, linewidth=2)
    ax.annotate(f'{np.median(y):.3f}', (i + 0.28, np.median(y)), va='center', color=TINTA, fontsize=10)
ax.axhline(0.5, color=TINTA2, linewidth=1, linestyle='--')
ax.set_xticks(range(3), [NOMBRE[b] for b in BRAZOS])
ax.set_ylabel('c-index en el fold de prueba')
ax.set_title(f'Desempeño fuera de fold ({ETIQUETA}, {len(pf) // 3} folds)', color=TINTA, loc='left')
plt.show()

## 2. Diferencias pareadas
Cada punto compara dos brazos en el mismo fold. A la derecha del 0 gana el primero.

In [ ]:
piv = pf.pivot_table(index=['repeticion', 'fold'], columns='brazo', values='cindex')
pares = [('coxnet', 'edad'), ('genetico', 'edad'), ('genetico', 'coxnet')]
textos = []
fig, ax = plt.subplots(figsize=(8, 3.8))
for i, (a, b) in enumerate(pares):
    dif = (piv[a] - piv[b]).to_numpy()
    ax.scatter(dif, i + rng.uniform(-0.12, 0.12, len(dif)), s=36, color=COLOR[a], alpha=0.8,
               edgecolor='white', linewidth=1)
    ax.vlines(dif.mean(), i - 0.25, i + 0.25, color=TINTA, linewidth=2)
    textos.append((i, f'media {dif.mean():+.3f}\ngana en {np.mean(dif > 0):.0%}'))
xmax = ax.get_xlim()[1]
for i, t in textos:   # resumen a la derecha, fuera de los puntos
    ax.annotate(t, (xmax, i), xytext=(8, 0), textcoords='offset points', va='center',
                color=TINTA, fontsize=9, annotation_clip=False)
ax.axvline(0, color=TINTA2, linewidth=1, linestyle='--')
ax.set_yticks(range(3), [f'{NOMBRE[a]} − {NOMBRE[b]}' for a, b in pares])
ax.set_xlabel('diferencia de c-index (mismo fold)')
ax.grid(axis='y', visible=False)
plt.show()

## 3. El optimismo del genético
Su aptitud (c-index en la CV interna) frente a su c-index real en la prueba. Si no sobreajustara, los puntos caerían sobre la diagonal; por debajo de ella, la aptitud promete más de lo que cumple.

In [ ]:
g = pf[pf.brazo == 'genetico']
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(g.c_interna, g.cindex, s=40, color=COLOR['genetico'], alpha=0.85, edgecolor='white', linewidth=1)
lo, hi = min(g.c_interna.min(), g.cindex.min()) - 0.02, max(g.c_interna.max(), g.cindex.max()) + 0.02
ax.plot([lo, hi], [lo, hi], color=TINTA2, linewidth=1, linestyle='--')
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel('aptitud interna (c-index, CV interna)'); ax.set_ylabel('c-index en la prueba')
ax.set_title(f'brecha media {g.brecha.mean():+.3f}', color=TINTA, loc='left')
plt.show()

## 4. Curva del genético
Mejor aptitud por generación: mediana de los folds y rango intercuartílico. Que siga subiendo no significa que generalice mejor (ver la sección 3).

In [ ]:
q = curvas.groupby('generacion')['aptitud_max'].quantile([0.25, 0.5, 0.75]).unstack()
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.fill_between(q.index, q[0.25], q[0.75], color=COLOR['genetico'], alpha=0.2, linewidth=0)
ax.plot(q.index, q[0.5], color=COLOR['genetico'], linewidth=2)
ax.axhline(g.cindex.mean(), color=TINTA2, linewidth=1, linestyle='--')
ax.annotate('c-index medio en prueba', (q.index.max(), g.cindex.mean()), ha='right', va='bottom',
            color=TINTA2, fontsize=9)
ax.set_xlabel('generación'); ax.set_ylabel('mejor aptitud interna')
plt.show()

## 5. Estabilidad de la selección
En cuántos de los 50 folds se eligió cada característica. Una selección estable repetiría las mismas.

In [ ]:
n_folds = pf.groupby(['repeticion', 'fold']).ngroups
fig, ejes = plt.subplots(2, 1, figsize=(8, 9))
for ax, b in zip(ejes, ['coxnet', 'genetico']):
    top = sel[sel.brazo == b].caracteristica.value_counts().head(12)[::-1]
    if top.empty:
        ax.text(0.5, 0.5, 'no eligió ninguna característica', ha='center', transform=ax.transAxes)
    else:
        ax.barh(top.index, top.values, color=COLOR[b], height=0.7)
        for y, v in enumerate(top.values):
            ax.annotate(str(v), (v, y), xytext=(3, 0), textcoords='offset points', va='center', fontsize=9)
    ax.set_xlim(0, n_folds)
    ax.set_xlabel(f'folds en que se eligió (de {n_folds})')
    ax.set_title(NOMBRE[b], color=TINTA, loc='left')
    ax.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

## 6. Todos los análisis
c-index medio por brazo en cada análisis disponible. Compara las máscaras predichas con las manuales, el efecto del filtro D26 y el de agregar la resección.

In [ ]:
filas = []
for p in sorted(BASE.iterdir()):
    if (p / 'por_fold.csv').exists():
        t = pd.read_csv(p / 'por_fold.csv').groupby('brazo')[['cindex', 'exactitud']].mean()
        filas.append({'analisis': p.name, **{f'c {NOMBRE[b]}': t.loc[b, 'cindex'] for b in BRAZOS},
                      **{f'exactitud {NOMBRE[b]}': t.loc[b, 'exactitud'] for b in BRAZOS}})
pd.DataFrame(filas).set_index('analisis').round(3)